# Churn model: training + Streamlit deployment
Run the cells from top to bottom. All files are written flat in this folder (next to the dataset CSV).

Files created: `features.py`, `train.py`, `app.py`, `requirements.txt`, then `churn_model.joblib` after training.

## 1. `features.py` (shared feature engineering)

In [ ]:
%%writefile features.py
"""Shared feature engineering. Imported by train.py AND app.py so the saved model loads cleanly."""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

NUMERIC_RAW = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
NUMERIC_ENGINEERED = ["tenure_monthly_charge_ratio", "high_spend_customer", "long_term_customer"]
NUMERIC_ALL = NUMERIC_RAW + NUMERIC_ENGINEERED

CATEGORICAL = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]
RAW_COLUMNS = NUMERIC_RAW + CATEGORICAL


class ChurnFeatureBuilder(BaseEstimator, TransformerMixin):
    """Cleans raw Telco rows and adds the engineered features from the notebook.

    Unlike the notebook, the 'high spend' median is learned in fit() (training data only),
    so there is no leakage and the same threshold is reused at prediction time.
    """

    def _clean(self, X):
        X = pd.DataFrame(X).copy()
        for col in RAW_COLUMNS:
            if col not in X.columns:
                X[col] = np.nan
        X = X[RAW_COLUMNS]

        # SeniorCitizen may arrive as 0/1 or Yes/No
        if X["SeniorCitizen"].dtype == object:
            X["SeniorCitizen"] = X["SeniorCitizen"].astype(str).str.strip().str.lower().map(
                {"yes": 1, "no": 0, "1": 1, "0": 0, "true": 1, "false": 0}
            )
        for col in NUMERIC_RAW:
            X[col] = pd.to_numeric(X[col], errors="coerce")  # TotalCharges has blank strings in Telco data

        # New customers (tenure 0) have no charges yet
        X.loc[X["TotalCharges"].isna() & (X["tenure"] == 0), "TotalCharges"] = 0.0

        for col in CATEGORICAL:
            s = X[col].astype(object)
            s = s.where(s.notna(), np.nan)
            X[col] = s.map(lambda v: v.strip() if isinstance(v, str) else v)
        return X

    def fit(self, X, y=None):
        X = self._clean(X)
        med = X["MonthlyCharges"].median()
        self.charge_median_ = float(med) if pd.notna(med) else 0.0
        return self

    def transform(self, X):
        X = self._clean(X)
        X["tenure_monthly_charge_ratio"] = X["tenure"] / (X["MonthlyCharges"] + 1e-6)
        X["high_spend_customer"] = (X["MonthlyCharges"] > self.charge_median_).astype(int)
        X["long_term_customer"] = (X["tenure"] >= 12).astype(int)
        return X[NUMERIC_ALL + CATEGORICAL]

## 2. `train.py` (compare 6 models, tune the best, save it)

In [ ]:
%%writefile train.py
"""Trains, compares, tunes and saves the best churn model -> churn_model.joblib"""
import argparse
import glob
import json
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (ExtraTreesClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     cross_val_predict, cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from features import CATEGORICAL, NUMERIC_ALL, RAW_COLUMNS, ChurnFeatureBuilder

warnings.filterwarnings("ignore")
SEED = 42
MODEL_PATH = "churn_model.joblib"
DEFAULT_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"


def find_csv(cli_path=None):
    candidates = [cli_path, DEFAULT_CSV,
                  "/mnt/c/Users/AJAY/Music/deployment/TELCO CUSTOMER CHURN/" + DEFAULT_CSV,
                  r"C:\Users\AJAY\Music\deployment\TELCO CUSTOMER CHURN\\" + DEFAULT_CSV]
    candidates += glob.glob("*elco*.csv") + glob.glob("*hurn*.csv")
    for c in candidates:
        if c and os.path.isfile(c):
            return c
    sys.exit(f"Dataset not found. Put '{DEFAULT_CSV}' next to this notebook or pass --data <path>.")


def make_preprocessor():
    num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUMERIC_ALL), ("cat", cat, CATEGORICAL)])


def make_pipeline(model):
    return Pipeline([("features", ChurnFeatureBuilder()), ("prep", make_preprocessor()), ("model", model)])


def candidate_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED),
        "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, class_weight="balanced", random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
        "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
        "Extra Trees": ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
    }


PARAM_GRIDS = {
    "Logistic Regression": {"model__C": [0.01, 0.05, 0.1, 0.5, 1, 5]},
    "Decision Tree": {"model__max_depth": [3, 4, 5, 6, 8], "model__min_samples_leaf": [10, 20, 40]},
    "Random Forest": {"model__max_depth": [6, 8, 12, None], "model__min_samples_leaf": [2, 5, 10], "model__max_features": ["sqrt", 0.3]},
    "Gradient Boosting": {"model__n_estimators": [100, 200], "model__learning_rate": [0.03, 0.05, 0.1], "model__max_depth": [2, 3, 4], "model__subsample": [0.8, 1.0]},
    "Extra Trees": {"model__max_depth": [8, 12, None], "model__min_samples_leaf": [2, 5, 10], "model__max_features": ["sqrt", 0.3]},
    "Hist Gradient Boosting": {"model__learning_rate": [0.03, 0.05, 0.1], "model__max_depth": [3, 5, None], "model__max_leaf_nodes": [15, 31], "model__l2_regularization": [0.0, 1.0]},
}


def best_threshold(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
    return float(thr[int(np.nanargmax(f1))])


def metrics_at(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0), "F1": f1_score(y_true, pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_true, proba), "Threshold": thr}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=None)
    ap.add_argument("--fast", action="store_true", help="skip hyper-parameter search")
    args = ap.parse_args()

    csv = find_csv(args.data)
    print("Loading:", csv)
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    missing = [c for c in RAW_COLUMNS + ["Churn"] if c not in df.columns]
    if missing:
        sys.exit(f"Dataset is missing columns: {missing}")

    df["Churn"] = df["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0, "1": 1, "0": 0})
    df = df.dropna(subset=["Churn"]).drop_duplicates()
    y = df["Churn"].astype(int)
    X = df[RAW_COLUMNS].copy()
    print(f"Rows: {len(X)} | churn rate: {y.mean():.3f}")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # 1) Compare all models (whole pipeline inside CV -> no leakage)
    rows = []
    for name, model in candidate_models().items():
        pipe = make_pipeline(model)
        auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
        f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
        rows.append({"Model": name, "CV_ROC_AUC": auc.mean(), "CV_ROC_AUC_std": auc.std(), "CV_F1": f1.mean()})
        print(f"{name:24s} CV ROC-AUC {auc.mean():.4f} (+/-{auc.std():.4f}) | CV F1 {f1.mean():.4f}")
    cv_df = pd.DataFrame(rows).sort_values("CV_ROC_AUC", ascending=False).reset_index(drop=True)
    winner = cv_df.loc[0, "Model"]
    print("\nBest model by CV ROC-AUC:", winner)

    # 2) Light hyper-parameter tuning of the winner
    pipe = make_pipeline(candidate_models()[winner])
    if not args.fast:
        search = RandomizedSearchCV(pipe, PARAM_GRIDS[winner], n_iter=12, scoring="roc_auc", cv=cv,
                                    random_state=SEED, n_jobs=1, refit=True, error_score="raise")
        search.fit(X_train, y_train)
        pipe = search.best_estimator_
        print("Tuned params:", search.best_params_, "| CV ROC-AUC:", round(search.best_score_, 4))
    else:
        pipe.fit(X_train, y_train)

    # 3) Pick decision threshold from out-of-fold predictions on TRAIN only
    oof = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    thr = best_threshold(y_train, oof)

    # 4) Honest evaluation on untouched test set
    pipe.fit(X_train, y_train)
    test_proba = pipe.predict_proba(X_test)[:, 1]
    test_metrics = metrics_at(y_test, test_proba, thr)
    default_metrics = metrics_at(y_test, test_proba, 0.5)
    print("\nTEST @ tuned threshold:", {k: round(v, 4) for k, v in test_metrics.items()})
    print("TEST @ 0.50 threshold :", {k: round(v, 4) for k, v in default_metrics.items()})

    # 5) Refit on ALL data for deployment, then save
    final = make_pipeline(candidate_models()[winner]).set_params(
        **{k: v for k, v in pipe.get_params().items() if k.startswith("model__")})
    final.fit(X, y)

    categories = {c: sorted(X[c].dropna().astype(str).str.strip().unique().tolist()) for c in CATEGORICAL}
    artifact = {
        "pipeline": final, "threshold": thr, "model_name": winner, "test_metrics": test_metrics,
        "categories": categories, "sklearn_version": sklearn.__version__,
        "tenure_max": int(X["tenure"].max()), "monthly_max": float(X["MonthlyCharges"].max()),
    }
    joblib.dump(artifact, MODEL_PATH, compress=3)
    cv_df.to_csv("model_comparison.csv", index=False)
    with open("model_metrics.json", "w") as f:
        json.dump({"model": winner, "threshold": thr, "test": test_metrics}, f, indent=2)
    print(f"\nSaved {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB), model_comparison.csv, model_metrics.json")


if __name__ == "__main__":
    main()

## 3. `app.py` (Streamlit app)

In [ ]:
%%writefile app.py
import os
import warnings

import joblib
import pandas as pd
import streamlit as st

warnings.filterwarnings("ignore")
st.set_page_config(page_title="Customer Churn Predictor", page_icon="📉", layout="wide")

MODEL_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "churn_model.joblib")

DEFAULT_CATS = {
    "gender": ["Female", "Male"], "Partner": ["No", "Yes"], "Dependents": ["No", "Yes"],
    "PhoneService": ["No", "Yes"], "MultipleLines": ["No", "No phone service", "Yes"],
    "InternetService": ["DSL", "Fiber optic", "No"],
    "OnlineSecurity": ["No", "No internet service", "Yes"], "OnlineBackup": ["No", "No internet service", "Yes"],
    "DeviceProtection": ["No", "No internet service", "Yes"], "TechSupport": ["No", "No internet service", "Yes"],
    "StreamingTV": ["No", "No internet service", "Yes"], "StreamingMovies": ["No", "No internet service", "Yes"],
    "Contract": ["Month-to-month", "One year", "Two year"], "PaperlessBilling": ["No", "Yes"],
    "PaymentMethod": ["Bank transfer (automatic)", "Credit card (automatic)", "Electronic check", "Mailed check"],
}
ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]


@st.cache_resource(show_spinner="Loading model...")
def load_artifact():
    if not os.path.exists(MODEL_PATH):
        return None, "Model file 'churn_model.joblib' not found. Run train.py first and keep it next to app.py."
    try:
        return joblib.load(MODEL_PATH), None
    except Exception as e:  # version mismatch, corrupt file, etc.
        return None, f"Could not load the model ({type(e).__name__}: {e}). Re-run train.py in the same environment as this app."


def predict_proba(artifact, frame):
    return artifact["pipeline"].predict_proba(frame)[:, 1]


def risk_label(p, thr):
    if p >= max(thr, 0.5) + 0.15 or p >= 0.75:
        return "🔴 High risk"
    if p >= thr:
        return "🟠 Elevated risk"
    return "🟢 Low risk"


artifact, load_error = load_artifact()
st.title("📉 Customer Churn Predictor")
st.caption("Telecom customer churn prediction")

if load_error:
    st.error(load_error)
    st.stop()

cats = {k: (artifact.get("categories", {}).get(k) or v) for k, v in DEFAULT_CATS.items()}
thr = float(artifact.get("threshold", 0.5))
tenure_max = max(72, int(artifact.get("tenure_max", 72)))
monthly_max = max(150.0, float(artifact.get("monthly_max", 120.0)))

with st.sidebar:
    st.subheader("Model info")
    st.write(f"**Model:** {artifact.get('model_name', 'unknown')}")
    st.write(f"**Decision threshold:** {thr:.2f}")
    m = artifact.get("test_metrics", {})
    if m:
        st.write(f"**Test ROC-AUC:** {m.get('ROC_AUC', float('nan')):.3f}")
        st.write(f"**Test Recall:** {m.get('Recall', float('nan')):.3f}")
        st.write(f"**Test Precision:** {m.get('Precision', float('nan')):.3f}")

tab_single, tab_batch = st.tabs(["Single customer", "Batch (CSV)"])

with tab_single:
    c1, c2, c3 = st.columns(3)
    with c1:
        st.markdown("**Customer**")
        gender = st.selectbox("Gender", cats["gender"])
        senior = st.selectbox("Senior citizen", ["No", "Yes"])
        partner = st.selectbox("Partner", cats["Partner"])
        dependents = st.selectbox("Dependents", cats["Dependents"])
        tenure = st.slider("Tenure (months)", 0, tenure_max, 12)
    with c2:
        st.markdown("**Services**")
        phone = st.selectbox("Phone service", cats["PhoneService"])
        if phone == "No":
            lines = "No phone service"
            st.selectbox("Multiple lines", [lines], disabled=True)
        else:
            lines = st.selectbox("Multiple lines", [o for o in cats["MultipleLines"] if o != "No phone service"] or ["No", "Yes"])
        internet = st.selectbox("Internet service", cats["InternetService"])
        addon_values = {}
        for a in ADDONS:
            if internet == "No":
                addon_values[a] = "No internet service"
            else:
                opts = [o for o in cats[a] if o != "No internet service"] or ["No", "Yes"]
                addon_values[a] = st.selectbox(a, opts, key=f"addon_{a}")
        if internet == "No":
            st.info("No internet: add-on services set to 'No internet service'.")
    with c3:
        st.markdown("**Billing**")
        contract = st.selectbox("Contract", cats["Contract"])
        paperless = st.selectbox("Paperless billing", cats["PaperlessBilling"])
        payment = st.selectbox("Payment method", cats["PaymentMethod"])
        monthly = st.number_input("Monthly charges", min_value=0.0, max_value=float(monthly_max), value=70.0, step=1.0)
        total = st.number_input("Total charges", min_value=0.0, value=float(round(monthly * tenure, 2)), step=10.0,
                                help="Defaults to monthly charges x tenure; edit if you know the exact value.")

    if st.button("Predict churn", type="primary", use_container_width=True):
        row = pd.DataFrame([{
            "gender": gender, "SeniorCitizen": 1 if senior == "Yes" else 0, "Partner": partner,
            "Dependents": dependents, "tenure": tenure, "PhoneService": phone, "MultipleLines": lines,
            "InternetService": internet, **addon_values, "Contract": contract,
            "PaperlessBilling": paperless, "PaymentMethod": payment,
            "MonthlyCharges": monthly, "TotalCharges": total,
        }])
        try:
            p = float(predict_proba(artifact, row)[0])
            st.divider()
            a, b = st.columns([1, 2])
            a.metric("Churn probability", f"{p:.1%}")
            b.markdown(f"### {risk_label(p, thr)}")
            b.progress(min(max(p, 0.0), 1.0))
            if p >= thr:
                st.warning("This customer is likely to churn. Consider a retention offer, e.g. a longer contract or discount.")
            else:
                st.success("This customer is unlikely to churn.")
        except Exception as e:
            st.error(f"Prediction failed: {e}")

with tab_batch:
    st.write("Upload a CSV with the same columns as the Telco dataset (`customerID` and `Churn` are optional).")
    up = st.file_uploader("CSV file", type=["csv"])
    if up is not None:
        try:
            data = pd.read_csv(up)
            data.columns = [c.strip() for c in data.columns]
            if data.empty:
                st.error("The file has no rows.")
            else:
                from features import RAW_COLUMNS
                absent = [c for c in RAW_COLUMNS if c not in data.columns]
                if absent:
                    st.error(f"Missing columns: {', '.join(absent)}")
                else:
                    probs = predict_proba(artifact, data[RAW_COLUMNS])
                    out = data.copy()
                    out["churn_probability"] = probs.round(4)
                    out["churn_prediction"] = (probs >= thr).map({True: "Yes", False: "No"})
                    st.success(f"Scored {len(out)} customers. {int((probs >= thr).sum())} flagged as likely to churn.")
                    st.dataframe(out.head(500), use_container_width=True)
                    st.download_button("Download results", out.to_csv(index=False).encode("utf-8"),
                                       "churn_predictions.csv", "text/csv")
        except Exception as e:
            st.error(f"Could not process the file: {e}")

## 4. `requirements.txt`

In [ ]:
# Writes requirements.txt with the EXACT versions installed in this environment.
# Pinning matters: the model is saved with this scikit-learn version, and Streamlit must load it with the same one.
from importlib.metadata import version, PackageNotFoundError

pkgs = ["streamlit", "scikit-learn", "pandas", "numpy", "joblib"]
fallback = {"streamlit": ">=1.30"}
lines = []
for p in pkgs:
    try:
        lines.append(f"{p}=={version(p)}")
    except PackageNotFoundError:
        lines.append(p + fallback.get(p, ""))
open("requirements.txt", "w").write("\n".join(lines) + "\n")
print(open("requirements.txt").read())

## 5. Train the model
Needs `WA_Fn-UseC_-Telco-Customer-Churn.csv` in this folder (or pass `--data path`). Takes a few minutes; add `--fast` to skip tuning.

In [ ]:
!python train.py

## 6. Look at the comparison

In [ ]:
import pandas as pd
pd.read_csv('model_comparison.csv')

## 7. Run the app locally
In a terminal (same environment, in this folder):

```
streamlit run app.py
```